In [7]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# Block 1: Load Data & The 91.35 Pseudo-Labels
# ------------------------------------------------------------------------------
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

# Load the file that scored your peak of 91.35
best_sub = pd.read_csv('submission_FINAL_ENSEMBLE.csv') 

test_pseudo = test.copy()
test_pseudo['demand'] = best_sub['demand']

# Combine into the unified dataset
combined_data = pd.concat([train, test_pseudo], axis=0).reset_index(drop=True)

# ------------------------------------------------------------------------------
# Block 2: The Clean 91.35 Base Features (No Noise)
# ------------------------------------------------------------------------------
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    parts = df['timestamp'].str.split(':', expand=True).astype(int)
    df['hour'] = parts[0]
    df['minute'] = parts[1]
    df['time_slot'] = df['hour'] * 4 + df['minute'] // 15
    
    df['sin_time'] = np.sin(2 * np.pi * df['time_slot'] / 96)
    df['cos_time'] = np.cos(2 * np.pi * df['time_slot'] / 96)
    df['time_slot_cat'] = 'Slot_' + df['time_slot'].astype(str)
    
    df['day_of_week'] = (df['day'] % 7).astype(str)
    df['is_weekend'] = df['day_of_week'].isin(['5', '6']).astype(str)
    
    df['geohash_5'] = df['geohash'].str[:5]
    df['geohash_4'] = df['geohash'].str[:4]
    
    # Internal Target Encoding Trigger
    df['geo_time'] = df['geohash'].astype(str) + "_" + df['time_slot_cat'].astype(str)
    
    return df

combined_data = build_features(combined_data)
test          = build_features(test)

# ------------------------------------------------------------------------------
# Block 3: Strict Categorical Typing
# ------------------------------------------------------------------------------
combined_data['Temperature'] = combined_data['Temperature'].fillna(-999)
test['Temperature']          = test['Temperature'].fillna(-999)

CAT_FEATURES = [
    'geohash', 'geohash_5', 'geohash_4', 'RoadType', 'LargeVehicles', 
    'Landmarks', 'Weather', 'time_slot_cat', 'day_of_week', 'is_weekend',
    'geo_time'
]

# Formatting for CatBoost (Strings) and LightGBM (Category)
for col in CAT_FEATURES:
    combined_data[col] = combined_data[col].fillna('Unknown').astype('category')
    test[col]          = test[col].fillna('Unknown').astype('category')

FEATURES = [
    'geohash', 'geohash_5', 'geohash_4', 'hour', 'minute', 'time_slot', 'time_slot_cat',
    'day_of_week', 'is_weekend', 'geo_time', 'RoadType', 'NumberofLanes', 
    'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'sin_time', 'cos_time'
]
TARGET = 'demand'

# Log compression
combined_data['log_demand'] = np.log1p(combined_data[TARGET])

# ------------------------------------------------------------------------------
# Block 4: Micro-Learning Rate Master Ensemble
# ------------------------------------------------------------------------------
print(f"Training Final Deep Ensemble on {len(combined_data)} rows...")

SEEDS = [42, 777, 2024]
test_preds_log = np.zeros(len(test))

for seed in SEEDS:
    print(f"\n--- Training Seed {seed} ---")
    
    # Model 1: CatBoost (Deep, Slow, Symmetric)
    print("  -> Fitting CatBoost (12,000 iterations)...")
    cb_model = CatBoostRegressor(
        iterations=12000,           # Massive depth for final squeeze
        learning_rate=0.015,        # Micro-learning rate prevents overfitting
        depth=8, 
        l2_leaf_reg=6,              # Higher L2 regularization for safety
        subsample=0.80,
        bootstrap_type='Bernoulli', 
        loss_function='RMSE', 
        random_seed=seed, 
        verbose=0
    )
    # Convert specifically back to string for CB backend
    cb_cat_data = combined_data.copy()
    cb_test_data = test.copy()
    for col in CAT_FEATURES:
        cb_cat_data[col] = cb_cat_data[col].astype(str)
        cb_test_data[col] = cb_test_data[col].astype(str)
        
    cb_model.fit(cb_cat_data[FEATURES], cb_cat_data['log_demand'], cat_features=CAT_FEATURES)
    cb_preds = cb_model.predict(cb_test_data[FEATURES])
    
    # Model 2: LightGBM (Deep, Slow, Leaf-wise)
    print("  -> Fitting LightGBM (12,000 iterations)...")
    lgb_model = lgb.LGBMRegressor(
        n_estimators=12000,
        learning_rate=0.01,         # Micro-learning rate
        num_leaves=63,
        subsample=0.80,
        colsample_bytree=0.80,
        reg_alpha=0.1,              # Added L1 regularization
        reg_lambda=5.0,             # Added L2 regularization
        random_state=seed,
        n_jobs=-1,
        verbose=-1
    )
    lgb_model.fit(combined_data[FEATURES], combined_data['log_demand'], categorical_feature=CAT_FEATURES)
    lgb_preds = lgb_model.predict(test[FEATURES])
    
    # Blend models: Favoring CatBoost slightly as it naturally handles categoricals better
    blended_seed_preds = (cb_preds * 0.6) + (lgb_preds * 0.4)
    test_preds_log += blended_seed_preds / len(SEEDS)

# ------------------------------------------------------------------------------
# Block 5: Reversal and Submission
# ------------------------------------------------------------------------------
final_test_preds = np.expm1(test_preds_log)

# Safe Correction Factor
final_test_preds = final_test_preds * 1.005 
final_test_preds = np.clip(final_test_preds, 0, None)

submission = pd.DataFrame({'Index': test['Index'], 'demand': final_test_preds})
submission.to_csv('submission_FINAL_DEEP.csv', index=False)
print("\nSuccess! Saved to submission_FINAL_DEEP.csv")

Training Final Deep Ensemble on 119077 rows...

--- Training Seed 42 ---
  -> Fitting CatBoost (12,000 iterations)...
  -> Fitting LightGBM (12,000 iterations)...

--- Training Seed 777 ---
  -> Fitting CatBoost (12,000 iterations)...
  -> Fitting LightGBM (12,000 iterations)...

--- Training Seed 2024 ---
  -> Fitting CatBoost (12,000 iterations)...
  -> Fitting LightGBM (12,000 iterations)...

Success! Saved to submission_FINAL_DEEP.csv
